In [1]:
################
# Tabular ARGN #
################

## Authors:             Ricardo Zambrano
## email:               rzambrano@gmail.com
## Date:               2026-03-14

## reference:           "TabularARGN: A flexible and Efficient Auto-Regressive Framework for Generating high-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/pdf/2501.12012

######################
# Tabular ARGN       #
######################

## Author:             Ricardo Zambrano
## Email:              rzambrano@gmail.com
## Date:               2026-03-14
## Reference:          "TabularARGN: A Flexible and Efficient Auto-Regressive Framework
##                      for Generating High-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/abs/[arxiv-id]  # fill this in

#################
# Session Setup #
#################

# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

from typing import Protocol

from datetime import date, datetime

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Experiment Tracking ───────────────────────────────────────────────────────
import wandb
import mlflow
from omegaconf import DictConfig, OmegaConf
import hydra

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Hardcoded Variables ────────────────────────────────────────────────────────

# -- None --


c:\Users\rzamb\Documents\tabular_nn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Working Directory ──────────────────────────────────────────────────────────

current_dir = Path.cwd().resolve()
print(current_dir)

target_dir = Path(r"c:\Users\rzamb\Documents\tabular_nn").resolve()

if current_dir != target_dir:

    # Get the path object for two levels up
    two_levels_up = current_dir.parents[0]

    # Change the working directory to the new path
    os.chdir(two_levels_up)

# Optional: Print new working directory to confirm the change
print(f"New working directory: {Path.cwd()}")

C:\Users\rzamb\Documents\tabular_nn\notebooks
New working directory: C:\Users\rzamb\Documents\tabular_nn


In [3]:
####################
# Helper Functions #
####################

# -- None --

In [4]:
####################
# Data Loading     #
####################

insurance_data = pd.read_csv("./data/raw/insurance/insurance.csv")

In [5]:
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
insurance_data.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
insurance_data.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [8]:
beijing_data = pd.read_csv("./data/raw/beijing/PRSA_Data_Dingling_20130301-20170228.csv")

In [9]:
beijing_data.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling


In [10]:
beijing_data["sample_date"] = pd.to_datetime({
    'year': beijing_data['year'],
    'month': beijing_data['month'],
    'day': beijing_data['day']
})

In [11]:
beijing_data = beijing_data.drop(columns=["year", "month", "day"])

In [12]:
beijing_data.dtypes

No                      int64
hour                    int64
PM2.5                 float64
PM10                  float64
SO2                   float64
NO2                   float64
CO                    float64
O3                    float64
TEMP                  float64
PRES                  float64
DEWP                  float64
RAIN                  float64
wd                     object
WSPM                  float64
station                object
sample_date    datetime64[ns]
dtype: object

In [13]:
####################
# Data Wrangling   #
####################

In [14]:
# Introducing a null values in different formats

insurance_data.iloc[0,0] = pd.NA
insurance_data.iloc[1,0] = np.nan
insurance_data.iloc[2,1] = None

# Inserting an artificial person with a number of children disconnected   from the sequence 1->5
insurance_data.iloc[3,3] = 14

# Rounding BMI to get a BINNED strategy
insurance_data["bmi"] = insurance_data["bmi"].round(1)

In [15]:
insurance_data.iloc[1317,:]

age              18.0
sex              male
bmi              53.1
children            0
smoker             no
region      southeast
charges     1163.4627
Name: 1317, dtype: object

In [16]:
df_pl = pl.from_pandas(insurance_data).clone()

In [17]:
df_pl[3,3] = 14

In [18]:
df_pl = df_pl.with_columns(
    pl.col("bmi").round(2)
)

In [19]:
# Inspecting how Polars casted missing values in a variety of formats

df_pl.head()

age,sex,bmi,children,smoker,region,charges
f64,str,f64,i64,str,str,f64
null,"""female""",27.9,0,"""yes""","""southwest""",16884.924
null,"""male""",33.8,1,"""no""","""southeast""",1725.5523
28.0,null,33.0,3,"""no""","""southeast""",4449.462
33.0,"""male""",22.7,14,"""no""","""northwest""",21984.47061
32.0,"""male""",28.9,0,"""no""","""northwest""",3866.8552


In [20]:
df_pl.null_count()

age,sex,bmi,children,smoker,region,charges
u32,u32,u32,u32,u32,u32,u32
2,1,0,0,0,0,0


In [21]:
df_pl["bmi"].drop_nulls().to_list()

[27.9,
 33.8,
 33.0,
 22.7,
 28.9,
 25.7,
 33.4,
 27.7,
 29.8,
 25.8,
 26.2,
 26.3,
 34.4,
 39.8,
 42.1,
 24.6,
 30.8,
 23.8,
 40.3,
 35.3,
 36.0,
 32.4,
 34.1,
 31.9,
 28.0,
 27.7,
 23.1,
 32.8,
 17.4,
 36.3,
 35.6,
 26.3,
 28.6,
 28.3,
 36.4,
 20.4,
 33.0,
 20.8,
 36.7,
 39.9,
 26.6,
 36.6,
 21.8,
 30.8,
 37.0,
 37.3,
 38.7,
 34.8,
 24.5,
 35.2,
 35.6,
 33.6,
 28.0,
 34.4,
 28.7,
 37.0,
 31.8,
 31.7,
 22.9,
 37.3,
 27.4,
 33.7,
 24.7,
 25.9,
 22.4,
 28.9,
 39.1,
 26.3,
 36.2,
 24.0,
 24.8,
 28.5,
 28.1,
 32.0,
 27.4,
 34.0,
 29.6,
 35.5,
 39.8,
 33.0,
 26.9,
 38.3,
 37.6,
 41.2,
 34.8,
 22.9,
 31.2,
 27.2,
 27.7,
 27.0,
 39.5,
 24.8,
 29.8,
 34.8,
 31.3,
 37.6,
 30.8,
 38.3,
 20.0,
 19.3,
 31.6,
 25.5,
 30.1,
 29.9,
 27.5,
 28.0,
 28.4,
 30.9,
 27.9,
 35.1,
 33.6,
 29.7,
 30.8,
 35.7,
 32.2,
 28.6,
 49.1,
 27.9,
 27.2,
 23.4,
 37.1,
 23.8,
 29.0,
 31.4,
 33.9,
 28.8,
 28.3,
 37.4,
 17.8,
 34.7,
 26.5,
 22.0,
 35.9,
 25.6,
 28.8,
 28.0,
 34.1,
 25.2,
 31.9,
 36.0,
 22.4,
 32.5,
 25.3,

In [22]:
nrow = df_pl.height

In [23]:
col_names = df_pl.columns

In [24]:
col_names

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [25]:
df_dtypes = [str(df_pl[name].dtype) for name in col_names]

In [26]:
df_dtypes

['Float64', 'String', 'Float64', 'Int64', 'String', 'String', 'Float64']

In [27]:
for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,dtype)

1 String
4 String
5 String


In [28]:
df_pl[:,1].unique().to_list()

['male', None, 'female']

In [29]:
categorical_encode_maps = {}

for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,col_names[i],df_pl[:,i].unique().to_list())

        unique_vals = df_pl[:, i].unique().to_list()
        col = col_names[i]

        if len(unique_vals) > nrow/3:
            Warning(f"Check {col} does not contain open ended values. This implementation only process categorical levels...")

        map_name = col + "_map"

        categorical_encode_maps[map_name] = {
            val: idx for idx, val in enumerate(unique_vals)
        }


1 sex [None, 'female', 'male']
4 smoker ['yes', 'no']
5 region ['southwest', 'northeast', 'southeast', 'northwest']


In [30]:
categorical_encode_maps

{'sex_map': {'male': 0, 'female': 1, None: 2},
 'smoker_map': {'no': 0, 'yes': 1},
 'region_map': {'northwest': 0,
  'northeast': 1,
  'southwest': 2,
  'southeast': 3}}

In [31]:
len(categorical_encode_maps)

3

In [32]:
categorical_decode_maps = {
    outer_key: {v: k for k, v in inner_dict.items()}
    for outer_key, inner_dict in categorical_encode_maps.items()
}

In [33]:
categorical_decode_maps

{'sex_map': {0: 'male', 1: 'female', 2: None},
 'smoker_map': {0: 'no', 1: 'yes'},
 'region_map': {0: 'northwest',
  1: 'northeast',
  2: 'southwest',
  3: 'southeast'}}

In [34]:
insurance_data["bmi"]

0       27.9
1       33.8
2       33.0
3       22.7
4       28.9
        ... 
1333    31.0
1334    31.9
1335    36.8
1336    25.8
1337    29.1
Name: bmi, Length: 1338, dtype: float64

In [35]:
import importlib
import src.utils.tabular_datasets as tabular_datasets
importlib.reload(tabular_datasets)
from src.utils.tabular_datasets import ArgnDataset

In [36]:
insurance_table_test = ArgnDataset(insurance_data, clip_cols = True)

In [37]:
dir(insurance_table_test)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_bool_columns',
 '_categorical_columns',
 '_column_binned_designs',
 '_datetime_columns',
 '_float_columns_to_cast_to_integer',
 '_incompatible_columns',
 '_is_protocol',
 '_is_runtime_protocol',
 '_numeric_strategy',
 '_numerical_binned_columns',
 '_numerical_digit_columns',
 '_numerical_discrete_columns',
 '_numerical_float_columns',
 '_raw_data',
 '_table',
 '_table_pd',
 'argn_preprocessing',
 'categorical_decoding_maps',
 'categorical_encoding_maps',
 'clip_cols',
 'col_names',
 'dtypes',

In [38]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
5,31.0,female,25.7,0,no,southeast,3756.62160
6,46.0,female,33.4,1,no,southeast,8240.58960
7,37.0,female,27.7,3,no,northwest,7281.50560
8,37.0,male,29.8,2,no,northeast,6406.41070
9,60.0,female,25.8,0,no,northwest,28923.13692


In [39]:
insurance_table_test.table.head(10)

,age,sex,bmi,children,smoker,region,charges
0,0,0,27.9,0,0,0,16884.92400
1,0,1,33.8,1,1,2,1725.55230
2,11,2,33.0,3,1,2,4449.46200
3,16,1,22.7,5,1,1,21984.47061
4,15,1,28.9,0,1,1,3866.85520
5,14,0,25.7,0,1,2,3756.62160
6,29,0,33.4,1,1,2,8240.58960
7,20,0,27.7,3,1,1,7281.50560
8,20,1,29.8,2,1,3,6406.41070
9,43,0,25.8,0,1,1,28923.13692


In [40]:
insurance_table_test._categorical_columns


[('sex', 1), ('smoker', 4), ('region', 5)]

In [41]:
c,i = zip(*insurance_table_test._categorical_columns)
print(c)
print(i)

('sex', 'smoker', 'region')
(1, 4, 5)


In [42]:
insurance_table_test._float_columns_to_cast_to_integer

[('age', 0)]

In [43]:
insurance_table_test._table

age,sex,bmi,children,smoker,region,charges
i64,i64,f64,i64,i64,i64,f64
0,0,27.9,0,0,0,16884.924
0,1,33.8,1,1,2,1725.5523
11,2,33.0,3,1,2,4449.462
16,1,22.7,5,1,1,21984.47061
15,1,28.9,0,1,1,3866.8552
…,…,…,…,…,…,…
33,1,31.0,3,1,1,10600.5483
1,0,31.9,0,1,3,2205.9808
1,0,36.8,0,1,2,1629.8335


In [44]:
insurance_table_test.numerical_discrete_encoding_maps

{'age': {None: 0,
  18: 1,
  19: 2,
  20: 3,
  21: 4,
  22: 5,
  23: 6,
  24: 7,
  25: 8,
  26: 9,
  27: 10,
  28: 11,
  29: 12,
  30: 13,
  31: 14,
  32: 15,
  33: 16,
  34: 17,
  35: 18,
  36: 19,
  37: 20,
  38: 21,
  39: 22,
  40: 23,
  41: 24,
  42: 25,
  43: 26,
  44: 27,
  45: 28,
  46: 29,
  47: 30,
  48: 31,
  49: 32,
  50: 33,
  51: 34,
  52: 35,
  53: 36,
  54: 37,
  55: 38,
  56: 39,
  57: 40,
  58: 41,
  59: 42,
  60: 43,
  61: 44,
  62: 45,
  63: 46,
  64: 47},
 'children': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}}

In [45]:
insurance_table_test._incompatible_columns

[]

In [46]:
columns_with_float, _ = zip(*insurance_table_test._numerical_float_columns)

In [47]:
small_numbers = insurance_table_test._table['bmi'].to_numpy()
big_bumbers = insurance_table_test._table['charges'].to_numpy()

In [48]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [49]:
insurance_table_test._numerical_float_columns

[('bmi', 2), ('charges', 6)]

In [50]:
lower = np.quantile(small_numbers, 0.01)
upper = np.quantile(small_numbers, 0.99)

print(f"lower = {lower} - upper = {upper}")

clipped = np.clip(small_numbers, lower, upper)

np.sort(small_numbers[np.logical_not(small_numbers == clipped)])

lower = 17.96031 - upper = 46.31906999999996


array([17.937, 17.937, 17.937, 17.937, 17.937, 17.937, 17.937, 17.937,
       17.937, 17.937, 17.937, 17.937, 17.937, 17.937, 46.389, 46.389,
       46.389, 46.389, 46.389, 46.389, 46.389, 46.389, 46.389, 46.389,
       46.389, 46.389, 46.389, 46.389])

In [51]:
def clip_columns(df_pl: pl.DataFrame, cols_to_process: list[tuple[str,int]], lower_quantile: float = 0.01, upper_quantile: float = 0.99) -> pl.DataFrame:
    """
    Clips columns with numeical values.

    Parameters:
    ----------

    df_pl: pl.DataFrame
        A polars data aframe
    cols_to_process: list[tuple[str,int]]
        Lis of columns with either integer or float values to beclipped
    lower_quantile: int, default_value = 0.01
        Lower quantile threshold. Values below this quantile are replaced 
        with the value at this quantile. 
        Setting the lower cutoff at 0.01 selects the value below
        which the lowest 1% of data points fall
    upper_quantile: int, default_value = 0.99
        Upper quantile threshold. Values above this quantile are replaced 
        with the value at this quantile.
        Setting the upper cutoff at 0.99 selects the value below
        which the lowest 99% of data points fall. It also means selecting
        a value above which the highest 1% of data points fall.

    Returns:
    -------

    processed_df : pl.DataFrame
        Polars data frame with specified columns clipped to the given quantile range
    """

    processed_df = df_pl
    cols_to_process_names, _ = zip(*cols_to_process)

    for col_name in cols_to_process_names:
        col_values = df_pl[col_name].to_numpy()
        lower = np.quantile(col_values, 0.01)
        upper = np.quantile(col_values, 0.99)
        processed_df.with_columns(pl.col("a").clip(lower, upper))

    return processed_df

    

In [52]:
test_clipping_df1 = pl.DataFrame({"a": [-50, 5, 50, None], "b": [150.92, 3, 12, 123]})
print(test_clipping_df1)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ -50  ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 50   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [53]:
test_clipping_df2 =test_clipping_df1.with_columns(pl.col("a").clip(1, 10))
print(test_clipping_df2)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ 1    ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 10   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [54]:
import warnings

In [55]:
from src.utils.argn_encoder_decoder import BinDesign

In [56]:
one_col = BinDesign(1,[2,3])
another_col = BinDesign(2,[4,5,6])
test_bin_mapping_list = [one_col, another_col]
test_bin_mapping_dict = {"one_col_key" : BinDesign(1,[2,3]), "another_col_key" : BinDesign(2,[4,5,6])}

In [57]:
test_bin_mapping_list[0].n_bins

1

In [58]:
test_bin_mapping_dict["one_col_key"].n_bins

1

In [59]:
def get_bin_designs(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]]) -> dict[str,BinDesign]:
    """
    Assumes a polars data frame and calculates the number of bins and the corresponding edges  
    required for BINNED encoding for each column in float_cols. Returns this design as a dict. 

    Parameters:
    ----------

    df_pl : pl.DataFrame
        A polars data frame 
    float_cols : list[tuple[str,int]]
        A list of (column_name, column_index) pairs. The columns
        are a subset of the existing columns in df_pl 

    Returns:
    -------

    columns_bin_designs : dict[str,BinDesign]
        A dictionary with column name as key and a BinDesign 
        as a corresponding value. BinDesign is a container for
        the number of bins and the edges for a given column
    """

    if len(float_cols) == 0:
        warnings.warn("get_n_bins() received an empty list as an argument...")
        return {}

    MAX_BINS = 100

    cols_to_process_names, _ = zip(*float_cols)

    columns_bin_designs = {}
    
    for col_name in cols_to_process_names:
        n_unique_vals = len(df_pl[col_name].unique().to_list())
        n_bins = min(MAX_BINS, n_unique_vals)
        percentiles = np.linspace(0, 100, n_bins + 1)
        edges = np.percentile(df_pl[col].drop_nulls().to_numpy(), percentiles) # .drop_nulls() to avoid TypeError due to propagation of nulls
        
        unique_edges = np.unique(edges)  # remove duplicates from highly skewed data
        edges_diff = len(edges) - len(unique_edges)
        if edges_diff > 0:
            edges = unique_edges
            n_bins = n_bins - edges_diff

        columns_bin_designs[col_name] = BinDesign(n_bins, edges)
    
    return columns_bin_designs




In [60]:
n_unq_vl = len(list(np.unique(small_numbers)))
n_bn = min(100,n_unq_vl)

In [61]:
np.quantile(small_numbers, np.linspace(0, 1, n_bn + 1))

array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2    , 37.419  , 38.     , 38.2 

In [62]:
percentiles = np.linspace(0, 100, n_bn + 1)  # n_bins+1 edges for n_bins intervals
edges = np.percentile(small_numbers, percentiles) # add .dropna()
edges


array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2    , 37.419  , 38.     , 38.2 

In [63]:
edges = np.unique(edges)  # remove duplicates from highly skewed data
edges

array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2    , 37.419  , 38.     , 38.2 

In [64]:
def select_numeric_strategy(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]]) -> list[str]:

    col_encoding_strategy = []

    for col in float_cols:

        numbers_to_analyze = df_pl[col].to_list()
        string_numbers = [str(abs(number)) for number in numbers_to_analyze] 
        splited_numbers = [tuple(str_number.split('.')) for str_number in string_numbers]
        len_int_and_dec = [(len(x), len(y)) for x, y in splited_numbers]
        
        max_decimal_places = max(x[1] for x in len_int_and_dec)

        max_num_digits = max((x[0]+x[1]) for x in len_int_and_dec)

        if max_decimal_places <=2:
            col_encoding_strategy.append("BINNED")
        elif max_num_digits > 6 or max_decimal_places > 3:
            col_encoding_strategy.append("DIGIT")
        else:
            col_encoding_strategy.append("BINNED")

    return col_encoding_strategy

In [65]:
columns_with_float

('bmi', 'charges')

In [66]:
select_numeric_strategy(insurance_table_test._table, columns_with_float)

['DIGIT', 'DIGIT']

In [67]:
# Numeric discrete

def generate_numeric_discrete_encoding_mappings():
    
    numerical_discrete_encoding_maps = {}
    numerical_discrete_columns = []

    for i,dtype in enumerate(df_dtypes):
        if dtype in ["Int8", "Int16", "Int32", "Int64", "UInt8", "UInt16", "UInt32", "UInt64"]:

            unique_vals = df_pl[:, i].unique().to_list()
            col = col_names[i]

In [68]:
# Numeric Binned

In [69]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [70]:
insurance_table_test._numeric_strategy.values()

dict_values(['BINNED', 'DIGIT'])

In [71]:
numbers_to_analyze_test = insurance_table_test._table["bmi"].drop_nulls().to_list()

string_numbers_test = [str(abs(number)) for number in numbers_to_analyze_test] 
splited_numbers_test = [tuple(str_number.split('.')) for str_number in string_numbers_test]
len_int_and_dec_test = [(len(x), len(y)) for x, y in splited_numbers_test]

max_decimal_places_test = max(x[1] for x in len_int_and_dec_test)

max_num_digits_test = max((x[0]+x[1]) for x in len_int_and_dec_test)

In [72]:
digits, decimals = zip(*splited_numbers_test)

digits_len = [len(x) for x in digits]
decials_len = [len(x) for x in digits]

print(max(digits_len))
print(max(decials_len))

2
2


In [73]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [74]:
binned_strategy_target_cols = [col_name for col_name, strategy in insurance_table_test._numeric_strategy.items() if strategy == "BINNED"]
digit_strategy_target_cols = [col_name for col_name, strategy in insurance_table_test._numeric_strategy.items() if strategy == "DIGIT"]


In [75]:
binned_strategy_target_cols

['bmi']

In [76]:
_numerical_binned_columns = [(col_name, col_index) for col_name, col_index in insurance_table_test._numerical_float_columns if col_name in binned_strategy_target_cols]
_numerical_digit_columns = [(col_name, col_index) for col_name, col_index in insurance_table_test._numerical_float_columns if col_name in digit_strategy_target_cols]


In [77]:
_numerical_digit_columns

[('charges', 6)]

In [78]:
insurance_table_test._numerical_float_columns

[('bmi', 2), ('charges', 6)]

In [83]:
insurance_table_test._column_binned_designs["bmi"]

BinDesign(n_bins=100, edges=array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2   

In [79]:
insurance_table_test.table.round(1).iloc[1317,:]

age            1.0
sex            1.0
bmi           46.4
children       0.0
smoker         1.0
region         2.0
charges     1253.0
Name: 1317, dtype: float64

In [80]:
# Numeric Digit

In [81]:
# Datetime